### Test for retrieving the internal matrix
Retrieving the step-by-step procedure for calculating GLCM matrix<br>

Work in progress<br>
by H.Nishiyama with GitHub Copilot<br>
2026-06-25<br>

In [1]:
import os
import numpy as np
from radiomics import featureextractor
from radiomics.glcm import RadiomicsGLCM
from radiomics import base
# import logging
import six
import SimpleITK as sitk
from radiomics import cMatrices
#=== ref. URLs for logging
# https://www.tohoho-web.com/python/logging.html
# https://docs.python.org/ja/dev/library/logging.html
# https://note.com/yukikkoaimanabi/n/n95df016c06f2
# https://qiita.com/FukuharaYohei/items/92795107032c8c0bfd19

In [2]:
dataDir = './'

imageName = sitk.ReadImage(os.path.join(dataDir, 'forRadiomicsTest.nrrd'))
maskName = sitk.ReadImage(os.path.join(dataDir, 'Segmentation.seg.nrrd'))

In [3]:
# GLCM計算
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.settings['distances'] = [1]  # 1ピクセル距離
extractor.settings['symmetricalGLCM'] = True  # 対称GLCM
extractor.settings['normalize'] = None  # 正規化
extractor.settings['label'] = 1  # ラベル1の領域を対象
extractor.settings['binWidth'] = 1  # ビン幅
# extractor.settings['resampledPixelSpacing'] = None  # リサンプリングなし
# extractor.settings['interpolator'] = 'sitkBSpline'  # 補間方法
# extractor.settings['verbose'] = True  # 詳細ログ出力
# extractor.settings['enableCExtensions'] = True  # C拡張を有効化
extractor.settings['force2D'] = True  # 2D強制なし
# extractor.settings['force2Ddimension'] = 0  # 2D強制時の次元指定（0: XY, 1: XZ, 2: YZ）
# extractor.settings['correctMask'] = True  # マスク補正を有効化



In [4]:

# features_dict = extractor.execute(imageName, maskName)

# または直接GLCMクラスを使用：
# glcm_calc = RadiomicsGLCM(imageName, maskName, **settings)
glcm_calc = RadiomicsGLCM(imageName, maskName, **extractor.settings)


In [5]:
# 計算前に確認
print("Ng (グレーレベル数):", glcm_calc.coefficients['Ng'])
print("GrayLevels:", glcm_calc.coefficients['grayLevels'])


Ng (グレーレベル数): 4
GrayLevels: [1 2 3 4]


In [6]:
glcm_calc.execute()  # GLCM計算を実行

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


{'Autocorrelation': array(4.11543367),
 'ClusterProminence': array(12.55974719),
 'ClusterShade': array(1.99023407),
 'ClusterTendency': array(2.00913519),
 'Contrast': array(1.84375),
 'Correlation': array(0.04252529),
 'DifferenceAverage': array(1.03380102),
 'DifferenceEntropy': array(1.74432286),
 'DifferenceVariance': array(0.75839982),
 'Id': array(0.60246599),
 'Idm': array(0.56409439),
 'Idmn': array(0.90980792),
 'Idn': array(0.81845238),
 'Imc1': array(-0.04887176),
 'Imc2': array(0.39657997),
 'InverseVariance': array(0.47266511),
 'JointAverage': array(2.01817602),
 'JointEnergy': array(0.10470409),
 'JointEntropy': array(3.53612608),
 'MCC': array(0.26153283),
 'MaximumProbability': array(0.18112245),
 'SumAverage': array(4.03635204),
 'SumEntropy': array(2.40180216),
 'SumSquares': array(0.9632213)}

In [7]:
imageArray = glcm_calc.imageArray
maskArray = glcm_calc.maskArray

In [8]:
distances = np.array(glcm_calc.settings['distances'])
Ng = glcm_calc.coefficients['Ng']
force2D = glcm_calc.settings.get('force2D', False)
force2Ddimension = glcm_calc.settings.get('force2Ddimension', 0)

#以降、GitHub Copilotが出してきた手順で無事に途中の行列を得られた<br>
#しかしながら個々の手順についてのトレースは未<br>
#詳細を確認後にコメントを入れ、さらに最終的な値の計算までのステップを追加予定<br>

In [9]:
P_raw, angles = cMatrices.calculate_glcm(
    imageArray, maskArray, distances, Ng, force2D, force2Ddimension
)

In [10]:
GrayLevels = glcm_calc.coefficients['grayLevels']
P_raw = P_raw[:, GrayLevels - 1, :, :]
P_raw = P_raw[:, :, GrayLevels - 1, :]

In [11]:
if glcm_calc.symmetricalGLCM:
    P_raw = P_raw + np.transpose(P_raw, (0, 2, 1, 3))

In [12]:
P_sum = np.nansum(P_raw, axis=3)
print("Raw summed GLCM shape:", P_sum.shape)
print("Raw summed GLCM matrix:\n", P_sum[0])

Raw summed GLCM shape: (1, 4, 4)
Raw summed GLCM matrix:
 [[50. 60. 26. 13.]
 [60. 66. 19. 20.]
 [26. 19.  4.  7.]
 [13. 20.  7. 10.]]


↑　ここまでの計算結果は一致しているが、<br>
 'JointAverage': array(2.01817602),<br>
 'JointEnergy': array(0.10470409),<br>
 'JointEntropy': array(3.53612608),<br>
は下記と不一致。<br>
エクセルでのシミュレーション結果<br>
Joint average	2.016666667<br>
Joint variance	0.964007937<br>
Joint entropy	3.606139303<br>


2026-06-26 12:17:35
理由が判明した
parameter記入するyamlでは
  weightingNorm: no_weighting
  # weightingNorm: None（デフォルト）
  # 各角度ごとに特徴量を計算し、最後に平均を取ります。
  # weightingNorm: no_weighting
  # 角度を統合した GLCM を先に作り、そこから特徴量を計算します。
にすれば解決した。
3D-Slicerでも同様の差がでていたので、3D-Slicerでのパラメータ設定を再確認予定